In [ ]:

pip install quick-sentiments==0.4.8


In [2]:
!pip show quick_sentiments #check for latest version

Name: quick-sentiments
Version: 0.4.8
Summary: Sentiment Analysis pipeline
Home-page: https://github.com/AlabhyaMe/Sentiments-Analysis
Author: Alabhya Dahal
Author-email: Alabhya Dahal <alabhya.dahal@gmail.com>
License: MIT License
Location: C:\Users\meala\anaconda3\envs\quicksentiment\Lib\site-packages
Requires: gensim, nltk, numpy, pandas, polars, scikit-learn, scipy, spacy, xgboost
Required-by: 


In [ ]:
import polars as pl

# here I have three python script I built to pre_process the data and running the pipeline
# you can find the code in the tools/preprocess.py file
# you can find  the code in the tools/pipeline.py file
# the pre_process function is used to clean the text data, there are various options available, please check the tools/preprocess.py file for details
# the run_pipeline function is used to run the sentimental analysis pipeline, it takes the training data and the vectorizer and machine learning methods as input, and returns the results
from quick_sentiments import pre_process_nltk
from quick_sentiments import pre_process_spacy
from quick_sentiments import run_pipeline
from quick_sentiments import make_predictions
from quick_sentiments import evaluate_performance

### Training Dataset


Name your data as Train.csv and place it in the Training Data folder. Or you can change the path in the code below.


In [4]:
# keep you training dataset in the training data folder
# this template uses csv files 
# column names can be set in Python but this template does not automatically update the column for the demo 
# however, the function will give you the option to tell column names for the text and label data

df_train = pl.read_csv("demo/training_data/train.csv",encoding='ISO-8859-1') 
print(f"Dataset shape: {df_train.shape[0]} rows and {df_train.shape[1]} columns")


Dataset shape: 162758 rows and 5 columns


### DEMO

In [5]:
df_train.head()
# randomly select only 10% of the data since the dataset is large
#RUN ONLY ONCE
df_train = df_train.sample(fraction=0.25, shuffle=True, seed=42) 
df_train.head(5)

movieid,reviewerName,isFrequentReviewer,reviewText,sentiment
str,str,bool,str,str
"""don_vito_corleone_willy_wonka_…","""Jacob Hansen Jr.""",true,"""An acceptably mindless sanctua…","""NEGATIVE"""
"""annie_hall_james_t._kirk_hiccu…","""Manuel Ramirez""",false,"""Although many shots are out of…","""POSITIVE"""
"""ellis_redding_legend_forrest_g…","""Jose Mccormick""",false,null,"""NEGATIVE"""
"""infinite_gandalf_the_grey""","""Heidi Wood""",false,"""10 Cloverfield Lane is an exci…","""POSITIVE"""
"""katniss_everdeen_darth_vader_n…","""Nicholas Park""",true,"""It's funny, fast, and charming…","""POSITIVE"""


The dataset is for training. The sentiments are already labeled. This will allow us to train a model that can predict sentiments on new data.


In [6]:
# you can use the pre_process function to clean the text data
response_column = "reviewText" # this is the column name for the text data, feel free to change it to your text column name
sentiment_column = "sentiment" # this is the column name for the sentiment data, feel free to change it to your sentiment column name


In [7]:
# make changes as necessary
# inside the map_elements, add  the parameters [pre_process(x, parameters_to_be_added)] and set it True/False if it differs from the defualt value
# check the tools/preprocess.py file for the parameters and their default values
# some of the parameters are remove_brackets, remove_stopwords, remove_punctuation, remove_numbers, remove_emojis, remove_urls, remove_html_tags, lemmatize, stem, lowercase
df_train = pre_process_nltk(df_train, text_column=response_column, new_column_name="cleaned_text_nltk")


In [8]:
df_train = pre_process_spacy(df_train, text_column=response_column, new_column_name="cleaned_text_spacy")

In [9]:
df_train.head()

movieid,reviewerName,isFrequentReviewer,reviewText,sentiment,cleaned_text_nltk,cleaned_text_spacy
str,str,bool,str,str,str,str
"""don_vito_corleone_willy_wonka_…","""Jacob Hansen Jr.""",true,"""An acceptably mindless sanctua…","""NEGATIVE""","""an acceptably mindless sanctua…","""an acceptably mindless sanctua…"
"""annie_hall_james_t._kirk_hiccu…","""Manuel Ramirez""",false,"""Although many shots are out of…","""POSITIVE""","""although many shots are out of…","""although many shots are out of…"
"""ellis_redding_legend_forrest_g…","""Jose Mccormick""",false,null,"""NEGATIVE""","""""",""""""
"""infinite_gandalf_the_grey""","""Heidi Wood""",false,"""10 Cloverfield Lane is an exci…","""POSITIVE""","""cloverfield lane is an excitin…","""cloverfield lane is an excitin…"
"""katniss_everdeen_darth_vader_n…","""Nicholas Park""",true,"""It's funny, fast, and charming…","""POSITIVE""","""it s funny fast and charming""","""it s funny fast and charming"""


In [ ]:
#### in this template, there are five+ text representation / vectorizer methods available 
#### in the function run_pipeline (in python cell below), we shall make use of this, write the words inside [ ] for the methods you want to use
#### 1. Bag of Words [BOW] 
#### 2. Term Frequency [tf]
#### 3. TF -IDF    [tfidf]
#### 4. Word Embedding using Word2Vec (you can use other packages with slight changes) [wv] 
         # Word Embedding uses defualt 300 values; this will take some time to run
#### 5. Glove (you can use other packages with slight changes) [glove_25,glove_50, glove_100, gl0ve_200]

In [ ]:
#### in this template, there are also five machine learning methods that can be used
#### 1. Logistic Regression [logit]
#### 2. Random forest (recommended) (rf)
#### 3. XGBoosting  [XGB](word embedding and XGBoost may take long time to complete, combination of both is not recommended in local machine)
#### 4. Naive Bayes [nb]
#### 5. Neural Network [nn] (this will take some time to run, and may run out of memory if the dataset is large, so be careful when using this method)
#### 6. Tensorflow/Keras [tf, tensorflow, keras] (this will take some time to run, and may run out of memory if the dataset is large, so be careful when using this method)


In [10]:
# this is the example of how to use the function
# you can change the vectorizer_name and model_name to the ones you want to use
# for now we will use word embedding and logistic regression
# write the name of your columns in the text_column_name and sentiment_column_name
# the text_column_name is the column name of the text data, and sentiment_column_name is

# run_pipeline function will return the dataframe with the vectorized text, vectorizer used  and the model
# it will also print the results of the model, including the accuracy and F1 score
# note, even without hyperparameter tuning, the model is getting over 70% accuracy in my test
# there may not be a need to perform hyperparameter tuning, but you can set perform_tuning to True if you want to do that

model = run_pipeline(
    vectorizer_name="wv", # BOW, tf, tfidf, wv,  glove_25,glove_50, glove_100, gl0ve_200,
    model_name="tf", # logit, rf, XGB, nb, nn, tf for tensorflow models .#XGB takes long time, can not recommend using it on normal case
    df=df_train,
    text_column_name="cleaned_text_nltk",  # this is the column name of the text data, 
    sentiment_column_name = "sentiment",
    perform_tuning = True, # make this true if you want to perform hyperparameter tuning, it will take longer time and 
                            # may run out of memory if the dataset is large,
    random_state=259
)

# missing values in the text data will be removed

--- Running Pipeline for Wv + Tf ---
1. Splitting data into train/test...
2. Vectorizing  dataset (X)...
Loading pre-trained word2vec-google-news-300 model (this may take a few minutes)...
Word2Vec model loaded.
Transforming test data using loaded Word2Vec model...
3. Training and predicting...
   - Building TF Model: Input Features = 300, Output Classes = 2
   - Starting TensorFlow training with GridSearchCV for hyperparameter tuning...
Fitting 3 folds for each of 8 candidates, totalling 24 fits

   - Best Hyperparameters found:
{'batch_size': 64, 'model__dropout_rate': 0.2, 'model__hidden_units': 128}
   - Generating predictions...
4. Evaluating model...

Classification Report:
              precision    recall  f1-score   support

    NEGATIVE       0.67      0.56      0.61      2658
    POSITIVE       0.80      0.87      0.83      5480

    accuracy                           0.77      8138
   macro avg       0.74      0.71      0.72      8138
weighted avg       0.76      0.77      

In [11]:
evaluate_performance(model["y_test"], model["y_prob"],positive_label=1)

{'best_roc_threshold': np.float64(0.6855),
 'best_pr_threshold': np.float32(0.3888),
 'decile_table':     Threshold  Accuracy  Precision  Recall     F1
 0         0.0     0.673      0.673   1.000  0.805
 1         0.1     0.714      0.704   0.992  0.824
 2         0.2     0.736      0.725   0.980  0.833
 3         0.3     0.751      0.744   0.960  0.838
 4         0.4     0.767      0.774   0.925  0.843
 5         0.5     0.767      0.802   0.869  0.834
 6         0.6     0.753      0.833   0.792  0.812
 7         0.7     0.711      0.885   0.657  0.754
 8         0.8     0.655      0.914   0.538  0.678
 9         0.9     0.572      0.959   0.380  0.545
 10        1.0     0.329      1.000   0.003  0.007}

In [12]:
## the model is a dictionary that contains the results of the model, including the accuracy and F1 score

# you can access the results using the keys of the dictionary
print("Vectorizer used: ", model["vectorizer_name"])
print("Model used: ", model["model_object"])
print("Accuracy: ", model["accuracy"])



Vectorizer used:  wv
Model used:  KerasClassifier(
	model=<function build_keras_model at 0x00000141841BF2E0>
	build_fn=None
	warm_start=False
	random_state=None
	optimizer=rmsprop
	loss=None
	metrics=None
	batch_size=64
	validation_batch_size=None
	verbose=0
	callbacks=None
	validation_split=0.0
	shuffle=True
	run_eagerly=False
	epochs=10
	model__input_dim=300
	model__num_classes=2
	class_weight=None
	model__dropout_rate=0.2
	model__hidden_units=128
)
Accuracy:  0.7673875645121652


### New Dataset for prediction
You can use the same format as the training dataset, but ensure that it contains the "Response" column for text data. The "Sentiment" column is optional for prediction datasets, as it will be generated by the model.
Make sure the dataset is saved in the "New Data" folder and is in CSV format.

In [13]:
new_data = pl.read_csv("demo/new_data/test.csv",encoding='ISO-8859-1') #keep your file here
print(new_data.shape)
new_data= new_data.sample(fraction=0.25, shuffle=True, seed=42)
print(new_data.shape)

(55315, 4)
(13828, 4)


In [14]:
new_data = pre_process_nltk(new_data, text_column=response_column, new_column_name="cleaned_text")
new_data.head()

movieid,reviewerName,isTopCritic,reviewText,cleaned_text
str,str,bool,str,str
"""spectacular_whirlwind_dazzling…","""Craig Lambert""",false,"""Cranston commands the screen, …","""cranston commands the screen b…"
"""astonish_katniss_everdeen_myri…","""Sara French""",true,"""All good stuff. Rock delivers …","""all good stuff rock delivers s…"
"""gandalf_michael_corleone_edwar…","""Mallory Chung""",false,"""Downey Jr. continues with his …","""downey jr continues with his h…"
"""epic_stardust_hermione_granger…","""Anna Camacho""",false,"""It's not a great film, but let…","""it s not a great film but let …"
"""ellis_redding_t-800_harry_pott…","""Cheryl Fuller""",false,"""RECOMMENDED ""Deliverance"" with…","""recommended deliverance with a…"


In [15]:
make_predictions(
    new_data=new_data,
    text_column_name="cleaned_text",  # this is the column name of the text data,
    prediction_column_name="sentiment_predictions",  # Optional custom name
    trained_results=model
)

movieid,reviewerName,isTopCritic,reviewText,cleaned_text,sentiment_predictions
str,str,bool,str,str,str
"""spectacular_whirlwind_dazzling…","""Craig Lambert""",false,"""Cranston commands the screen, …","""cranston commands the screen b…","""NEGATIVE"""
"""astonish_katniss_everdeen_myri…","""Sara French""",true,"""All good stuff. Rock delivers …","""all good stuff rock delivers s…","""POSITIVE"""
"""gandalf_michael_corleone_edwar…","""Mallory Chung""",false,"""Downey Jr. continues with his …","""downey jr continues with his h…","""POSITIVE"""
"""epic_stardust_hermione_granger…","""Anna Camacho""",false,"""It's not a great film, but let…","""it s not a great film but let …","""POSITIVE"""
"""ellis_redding_t-800_harry_pott…","""Cheryl Fuller""",false,"""RECOMMENDED ""Deliverance"" with…","""recommended deliverance with a…","""POSITIVE"""
…,…,…,…,…,…
"""katniss_everdeen_superman_harr…","""Bryan Phillips""",true,"""""No one's riding that loco thi…","""no one s riding that loco thin…","""POSITIVE"""
"""evoke_wonder_woman_myriad_john…","""Michele Tucker""",true,"""[A Taste of Honey] has an eart…","""has an earthy gusto and sincer…","""POSITIVE"""
"""miracle_luke_skywalker_destiny…","""William Holland""",true,"""You put up with lines such as …","""you put up with lines such as …","""NEGATIVE"""
